# Exam project - Programming for Economists, summer 2026

Solution to the three exam problems. Problems 2 and 3 use the modules `SolowModel.py` and `PortfolioModel.py`; the notebook calls them, reports the numbers and makes the figures. Problem 1 is done in the notebook, with the state list and region mapping in `states.py`.

Structure:

1. Real GDP across US states (data, from FRED)
2. The Solow model with a time-varying savings rate
3. A portfolio with a risky and a safe asset

Uses only numpy, scipy, pandas and matplotlib, plus `fredapi` for the download in Problem 1.

In [ ]:
import numpy as np
import pandas as pd
from scipy import optimize

import matplotlib.pyplot as plt
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
plt.rcParams.update({'axes.grid':True,'grid.color':'black','grid.alpha':0.25,'grid.linestyle':'--'})
plt.rcParams.update({'font.size':12})

%load_ext autoreload
%autoreload 2

# folder of this notebook, so fredapi.txt and data/ are found
# no matter which working directory the notebook is run from
from pathlib import Path
HERE = Path.cwd()

# 1. Real GDP across US states

For each state we use two FRED series: `XXRGSP` (real GDP, millions of chained 2017 dollars, annual) and `XXPOP` (population, thousands of persons). The state codes and region mapping come from `states.py`.

### Requirements

Problem 1 downloads from FRED with the `fredapi` package (Lecture 11). Run the cell below
once to install it, and place a file `fredapi.txt` next to this notebook holding a FRED API
key from https://fredaccount.stlouisfed.org/apikey. If neither is available the notebook
falls back on the cached copy in `data/`.

In [ ]:
# run once if fredapi is not installed yet
#%pip install fredapi

In [ ]:
from states import STATES, REGION

print(f'{len(STATES)} states, e.g. {STATES[:5]}')
print(f"example region lookup: REGION['MS'] = {REGION['MS']}")

## 1.1 Question 1

### Download the data

We download `XXRGSP` and `XXPOP` for all 50 states into one data frame each, with years in the index and states in the columns. The API key is read from `fredapi.txt` (as in Lecture 11). If no key is available we fall back on the cached copy in the `data` folder.

In [ ]:
def download_fred(states, suffix):
    ''' download the series XX+suffix for every state code XX into one data frame '''

    from fredapi import Fred
    with open(HERE/'fredapi.txt','r') as f: fred = Fred(api_key=f.read().strip())

    data = {}
    for xx in states:
        s = fred.get_series(f'{xx}{suffix}')       # a pandas Series indexed by date
        s.index = s.index.year                      # keep the year only (annual data)
        data[xx] = s

    df = pd.DataFrame(data)                          # years in the index, states in columns
    df = df.rename_axis('year')
    return df

def load_cached(suffix):
    ''' fall back on the cached csv in ./data if FRED is not reachable '''
    df = pd.read_csv(HERE/'data'/f'{suffix}.csv', index_col=0)
    df.index = df.index.astype(int) # years were saved as strings in the csv header
    df = df.rename_axis('year')
    return df

try:
    rgsp = download_fred(STATES,'RGSP')             # real GDP, millions of $
    pop  = download_fred(STATES,'POP')              # population, thousands
    source = 'FRED (live download)'
except Exception as e:
    print(f'FRED download not available ({type(e).__name__}), using the cached copy')
    rgsp = load_cached('RGSP')
    pop  = load_cached('POP')
    source = 'cached copy in ./data'

print(f'data source: {source}')
print(f'RGSP: {rgsp.shape[0]} years x {rgsp.shape[1]} states, years {rgsp.index.min()}-{rgsp.index.max()}')
print(f'POP : {pop.shape[0]} years x {pop.shape[1]} states, years {pop.index.min()}-{pop.index.max()}')

### Keep the common years and complete states

The two series do not cover the same years, so we keep only the years present in both. We then drop any state with a missing observation in those years, so the panel is balanced.

In [ ]:
# a. common years
years = rgsp.index.intersection(pop.index)
rgsp = rgsp.loc[years]
pop  = pop.loc[years]

# b. drop states missing anywhere in either series
ok = rgsp.notna().all(axis=0) & pop.notna().all(axis=0)
rgsp = rgsp.loc[:,ok]
pop  = pop.loc[:,ok]

t_first, t_last = int(years.min()), int(years.max())
print(f'left with {len(years)} years ({t_first}-{t_last}) and {rgsp.shape[1]} states')
if (~ok).any(): print('dropped:', list(ok.index[~ok]))

### Real GDP per person

`XXRGSP` is in millions of dollars and `XXPOP` is in thousands of persons, so

$$ y_{i,t} = \frac{\text{RGSP}\times 10^6}{\text{POP}\times 10^3} = \frac{\text{RGSP}}{\text{POP}}\times 10^3 $$

gives GDP per person in dollars.

In [ ]:
y = rgsp/pop*1e3   # real GDP per person in dollars, years x states
y.head()

### Highest and lowest, in the first and the last year

In [ ]:
def hi_lo(row):
    return row.idxmax(), row.max(), row.idxmin(), row.min()

for t in [t_first,t_last]:
    hs,hv,ls,lv = hi_lo(y.loc[t])
    print(f'{t}: highest {hs} ${hv:,.0f}, lowest {ls} ${lv:,.0f}, '
          f'ratio {hv/lv:.2f}, average ${y.loc[t].mean():,.0f}')

### Figure: levels and relative to the average

In [ ]:
pick = ['NY','CA','TX','MS','WV']           # a handful of states of our own choice
avg = y.mean(axis=1)                          # average across states, per year

fig,axes = plt.subplots(1,2,figsize=(13,5))

# i. levels for the chosen states plus the cross-state average
ax = axes[0]
for st in pick: ax.plot(y.index, y[st], label=st)
ax.plot(y.index, avg, color='black', lw=2.5, label='average')
ax.set_title('Real GDP per person'); ax.set_xlabel('year'); ax.set_ylabel('$ per person')
ax.legend(ncol=2, fontsize=10)

# ii. every state relative to the cross-state average in the same year
ax = axes[1]
rel = y.div(avg, axis=0)
for st in y.columns: ax.plot(rel.index, rel[st], color=colors[0], alpha=0.25, lw=0.8)
ax.axhline(1.0, color='black', lw=1.5)
ax.set_title('Relative to the cross-state average'); ax.set_xlabel('year'); ax.set_ylabel('ratio to average')

fig.tight_layout()

The richer states (NY, CA) stay above the poorer ones (MS, WV) across the whole sample. The right panel shows the distribution around the average; the spread does not clearly narrow or widen by eye, which Question 2 measures directly.

## 1.2 Question 2

### Dispersion over time

For each year we compute the standard deviation across states of $\log y_{i,t}$. This is a
standard measure of how spread out the states are (sigma-convergence).

In [ ]:
logy = np.log(y)
sd = logy.std(axis=1, ddof=1)   # std across states, one number per year

fig,ax = plt.subplots(figsize=(9,5))
ax.plot(sd.index, sd.values, color=colors[0], lw=2)
ax.set_title('Standard deviation across states of log GDP per person')
ax.set_xlabel('year'); ax.set_ylabel('std of log y')
fig.tight_layout()

In [ ]:
imax, imin = sd.idxmax(), sd.idxmin()
print(f'{t_first}: {sd.loc[t_first]:.4f}')
print(f'{t_last}: {sd.loc[t_last]:.4f}')
print(f'highest in {imax}: {sd.loc[imax]:.4f}')
print(f'lowest  in {imin}: {sd.loc[imin]:.4f}')

### What the figure shows

The dispersion in log GDP per person is close to flat over the sample. The first-year and last-year values above show the direction: a last value below the first means the states grew slightly more alike (weak sigma-convergence), above means slightly more unequal. The change is small either way.

## 1.3 Question 3

### Growth and convergence

For each state the average annual growth rate is

$$ g_i = \frac{\log y_{i,t_{last}} - \log y_{i,t_{first}}}{t_{last}-t_{first}} $$

We fit $g_i = a + b\log y_{i,t_{first}}$ with `np.polyfit`. A negative slope $b$ is
beta-convergence: states that started poorer grew faster. From the slope,

$$ \lambda = -\frac{\log(1+bT)}{T}, \qquad h = \frac{\log 2}{\lambda}, \qquad T=t_{last}-t_{first} $$

In [ ]:
T = t_last - t_first
logy0 = logy.loc[t_first]                        # log y in the first year, per state
g = (logy.loc[t_last]-logy.loc[t_first])/T       # average annual growth, per state

b,a = np.polyfit(logy0, g, 1)                    # polyfit returns [slope, intercept]
corr = np.corrcoef(logy0, g)[0,1]
print(f'a = {a:.4f}, b = {b:.4f}, corr(g, log y0) = {corr:.4f}')

In [ ]:
lam = -np.log(1+b*T)/T
h = np.log(2)/lam
print(f'lambda = {lam:.4f} per year  (speed of convergence)')
print(f'h      = {h:.1f} years        (half-life of an initial gap)')

### Scatter with the fitted line and the extreme states

In [ ]:
fig,ax = plt.subplots(figsize=(10,6))
ax.scatter(logy0, g, color=colors[0], alpha=0.7)

xline = np.linspace(logy0.min(), logy0.max(), 100)
ax.plot(xline, a+b*xline, color='black', lw=2, label=f'fit: g = {a:.3f} + {b:.3f} log y0')

# label the five fastest and five slowest growing states
top5 = g.sort_values().index[-5:]
bot5 = g.sort_values().index[:5]
for st in list(top5)+list(bot5):
    ax.annotate(st, (logy0[st], g[st]), fontsize=10,
                xytext=(4,4), textcoords='offset points')

ax.set_title('Convergence: growth vs initial level')
ax.set_xlabel('log GDP per person in {}'.format(t_first)); ax.set_ylabel('average annual growth')
ax.legend()
fig.tight_layout()

### What the figure shows

The slope $b$ is negative: states that were poorer in the first year grew faster, so there is beta-convergence. The correlation above measures how tightly the points follow the line. $\lambda$ and $h$ translate the slope into the speed at which gaps close.

## 1.4 Question 4

### Regions

Using `REGION` and a groupby, for each year and each region we compute the average across the
states in that region of $y_{i,t}$ divided by the cross-state average in the same year. So a
value above 1 means the region is richer than the country in that year.

In [ ]:
region_of = pd.Series({st:REGION[st] for st in y.columns}, name='region')
print('states per region:')
print(region_of.value_counts().to_string())

In [ ]:
rel = y.div(y.mean(axis=1), axis=0)              # each state relative to the yearly average
region_rel = rel.T.groupby(region_of).mean().T     # average within region, per year -> years x 4 regions
region_rel.head()

In [ ]:
fig,ax = plt.subplots(figsize=(10,6))
for reg in region_rel.columns: ax.plot(region_rel.index, region_rel[reg], label=reg, lw=2)
ax.axhline(1.0, color='black', lw=1)
ax.set_title('Regions relative to the national average')
ax.set_xlabel('year'); ax.set_ylabel('ratio to average')
ax.legend()
fig.tight_layout()

In [ ]:
tab = pd.DataFrame({
    t_first: region_rel.loc[t_first],
    t_last:  region_rel.loc[t_last],
})
tab['change'] = tab[t_last]-tab[t_first]
display(tab.style.format('{:.3f}').set_caption('Region relative to national average'))

mover = tab['change'].abs().idxmax()
print(f'largest move: {mover} by {tab.loc[mover,"change"]:+.3f} '
      f'({"up" if tab.loc[mover,"change"]>0 else "down"})')

# 2. The Solow model with a time-varying savings rate

All variables are per worker. Output is $y_t=k_t^\alpha$, a share $s_t$ is invested, and
capital accumulates as $k_{t+1}=s_t f(k_t)+(1-\delta)k_t$. The savings rule is
$s_t=\bar s+(s_0-\bar s)\varphi^t$. Welfare is $W=\sum_{t=0}^{T-1}\beta^t\log c_t$.

The class `SolowModelClass` in `SolowModel.py` holds the parameters and the methods. We filled
in `s_path`, `welfare` and `evaluate`; the simulator and steady-state helpers were given.

In [ ]:
from SolowModel import SolowModelClass
model = SolowModelClass()
print(model)

## 2.1 Question 1

### Steady state and the baseline

First the steady state for $s=\bar s$ from the formulas, checked against a root-finder for
$k_{t+1}-k_t=0$.

In [ ]:
k_ss,y_ss,c_ss = model.steady_state()
k_ss_num = model.solve_steady_state()
print(f'analytical: k* = {k_ss:.6f}, y* = {y_ss:.6f}, c* = {c_ss:.6f}')
print(f'root-finder: k* = {k_ss_num:.6f}  (diff {abs(k_ss-k_ss_num):.1e})')

### The baseline simulation

The baseline is a constant savings rate $s_t=\bar s$ from $k_0=0.10$.

In [ ]:
baseline = model.simulate(model.par.s_bar)

fig,axes = plt.subplots(1,3,figsize=(14,4))
for ax,var,name in zip(axes,[baseline.s,baseline.k,baseline.c],['$s_t$','$k_t$','$c_t$']):
    ax.plot(var, color='black', lw=2)
    ax.set_title(name); ax.set_xlabel('t')
fig.suptitle('Baseline: constant savings rate', y=1.03)
fig.tight_layout()

print(f'c0 = {baseline.c[0]:.6f}, c_(T-1) = {baseline.c[-1]:.6f}, k_(T-1) = {baseline.k[-1]:.6f}')

### assert tests

Two cases where the answer is known: an economy that starts in $k^*$ must stay there, and with
$s_t=0$ capital decays as $k_t=(1-\delta)^t k_0$.

In [ ]:
# i. starting in k* stays in k*
sim_ss = model.simulate(model.par.s_bar, k0=k_ss)
assert np.allclose(sim_ss.k, k_ss), 'an economy starting in k* should stay in k*'

# ii. with s=0 capital decays geometrically
sim_zero = model.simulate(0.0)
t = np.arange(model.par.T)
assert np.allclose(sim_zero.k, (1-model.par.delta)**t * model.par.k0), 's=0 should give geometric decay'

print('both assert tests pass')

## 2.2 Question 2

### Four savings rules

The rule $s_t=\bar s+(s_0-\bar s)\varphi^t$ is in `s_path`. We simulate the four combinations
and plot them with the baseline.

In [ ]:
combos = [(0.30,0.50),(0.40,0.80),(0.10,0.50),(0.60,0.60)]

sims = {'baseline': baseline}
for s0,phi in combos:
    sims[f's0={s0}, phi={phi}'] = model.simulate(model.s_path(s0,phi))

fig,axes = plt.subplots(1,3,figsize=(15,4.5))
for ax,attr,name in zip(axes,['s','k','c'],['$s_t$','$k_t$','$c_t$']):
    for label,sim in sims.items():
        ax.plot(getattr(sim,attr), lw=2.5 if label=='baseline' else 1.5,
                color='black' if label=='baseline' else None, label=label)
    ax.set_title(name); ax.set_xlabel('t')
axes[0].legend(fontsize=9)
fig.tight_layout()

In [ ]:
print(f'{"rule":<20s}{"k_(T-1)":>12s}')
print(f'{"baseline":<20s}{baseline.k[-1]:>12.6f}')
for s0,phi in combos:
    kT = model.simulate(model.s_path(s0,phi)).k[-1]
    print(f'{f"s0={s0}, phi={phi}":<20s}{kT:>12.6f}')

### What s0 and phi do

All five paths end at the same $k_{T-1}$: the long-run capital stock depends only on the
long-run savings rate $\bar s$, which is the same in every rule, so the terminal capital is
common. $s_0$ sets where savings (and therefore investment) start: a high $s_0$ means high
early investment, so consumption is sacrificed early and capital is built up faster. $\varphi$
controls how long that initial deviation lasts: with $\varphi$ near 0 the rule snaps back to
$\bar s$ almost at once, while $\varphi$ near 1 keeps $s_t$ away from $\bar s$ for many
periods. So $s_0$ is the size of the initial push and $\varphi$ is its persistence.

## 2.3 Question 3

### Welfare of the rules

$W$ is in `welfare`, and `evaluate(s0,phi)` builds the path, simulates and returns $W$.

In [ ]:
W_base = model.evaluate(model.par.s_bar, 0.5)   # phi is irrelevant when s0 = s_bar

rows = []
for s0,phi in combos:
    W = model.evaluate(s0,phi)
    rows.append({'s0':s0,'phi':phi,'W':W,'W - W_base':W-W_base})

wtab = pd.DataFrame(rows)
display(wtab.style.format({'s0':'{:.2f}','phi':'{:.2f}','W':'{:.4f}','W - W_base':'{:+.4f}'})
        .set_caption(f'Welfare of the four rules (baseline W = {W_base:.4f})'))

In [ ]:
better = wtab[wtab['W - W_base']>0]
print('rules that beat the baseline:')
print(better[['s0','phi']].to_string(index=False) if len(better) else '  none')

### Could you have seen it from the Question 2 figure?

Only partly. The baseline starts far below the steady state (k0 = 0.10 against k* = 0.76), so the economy is capital poor and the return to early investment is high. Of the four rules only s0 = 0.30 beats the baseline: it saves a little more than 0.25 at the start, which builds capital faster and raises consumption in almost every later period. The low-s0 rule (s0 = 0.10) does worse, because cutting early investment when capital is scarce holds consumption down, and the losses come early so discounting does not offset them. The high-s0 rules (0.40, 0.60) oversave and give up too much early consumption. The c_t panel shows the ordering of early consumption, but the discounted sum cannot be read off by eye, so the welfare table is needed.

## 2.4 Question 4

### Grid then optimizer

We compute $W$ on a grid over $s_0\in[0,0.60]$ and $\varphi\in[0,0.95]$, illustrate it with
`contourf`, take the best grid point and polish it with a numerical optimizer.

In [ ]:
s0_grid  = np.linspace(0.0,0.60,61)
phi_grid = np.linspace(0.0,0.95,58)
W_grid = np.empty((s0_grid.size, phi_grid.size))
for i,s0 in enumerate(s0_grid):
    for j,phi in enumerate(phi_grid):
        W_grid[i,j] = model.evaluate(s0,phi)

bi,bj = np.unravel_index(np.argmax(W_grid), W_grid.shape)
s0_best, phi_best, W_best = s0_grid[bi], phi_grid[bj], W_grid[bi,bj]
print(f'best grid point: s0 = {s0_best:.3f}, phi = {phi_best:.3f}, W = {W_best:.4f}')

In [ ]:
fig,ax = plt.subplots(figsize=(9,6))
cs = ax.contourf(phi_grid, s0_grid, W_grid, levels=30)
fig.colorbar(cs, label='W')
ax.plot(phi_best, s0_best, '*', ms=18, color='white', mec='black', mew=1.2, label='best on grid')
ax.set_title('Welfare over (s0, phi)')
ax.set_xlabel('phi'); ax.set_ylabel('s0'); ax.legend()
fig.tight_layout()

In [ ]:
obj = lambda x: -model.evaluate(x[0], x[1])      # minimize minus welfare
res = optimize.minimize(obj, np.array([s0_best,phi_best]),
                        method='Nelder-Mead', options={'xatol':1e-6,'fatol':1e-10})
s0_opt, phi_opt = res.x
print(f'optimizer: s0 = {s0_opt:.4f}, phi = {phi_opt:.4f}, W = {-res.fun:.4f}')
print(f'grid:      s0 = {s0_best:.4f}, phi = {phi_best:.4f}, W = {W_best:.4f}')
print(f'improvement over the grid: {-res.fun - W_best:+.5f}')

The optimizer lands close to the best grid point and improves $W$ only slightly. The grid was already fine, so the optimizer only refines the value between grid lines.

In [ ]:
best_rule = model.simulate(model.s_path(s0_opt,phi_opt))

fig,axes = plt.subplots(1,3,figsize=(14,4))
for ax,attr,name in zip(axes,['s','k','c'],['$s_t$','$k_t$','$c_t$']):
    ax.plot(getattr(baseline,attr), color='black', lw=2, label='baseline')
    ax.plot(getattr(best_rule,attr), color=colors[1], lw=2, label='best rule')
    ax.set_title(name); ax.set_xlabel('t')
axes[0].legend(fontsize=10)
fig.tight_layout()

above = np.where(best_rule.c > baseline.c)[0]
print(f'c0 under the best rule = {best_rule.c[0]:.6f} (baseline {baseline.c[0]:.6f})')
print(f'first period where c_t is above the baseline: t = {above[0] if above.size else "never"}')

The best rule starts with a savings rate slightly above the baseline, not below it. Because the economy starts capital poor, saving a little more in the first periods builds capital faster; the small drop in consumption at t = 0 is offset by higher consumption in essentially every later period. The gain comes from a small early investment, after which the rule settles back to the baseline savings rate.

## 2.5 Question 5

### A free long-run level

Now $s_\infty$ is a third parameter: $s_t=s_\infty+(s_0-s_\infty)\varphi^t$. `evaluate` already takes an optional `s_inf`, so we optimize over all three.

In [ ]:
obj3 = lambda x: -model.evaluate(x[0], x[1], s_inf=x[2])
x0 = np.array([s0_opt, phi_opt, model.par.s_bar])
res3 = optimize.minimize(obj3, x0, method='Nelder-Mead',
                         options={'xatol':1e-6,'fatol':1e-10})
s0_3, phi_3, sinf_3 = res3.x
W3 = -res3.fun
print(f'three-parameter rule: s0 = {s0_3:.4f}, phi = {phi_3:.4f}, s_inf = {sinf_3:.4f}, W = {W3:.4f}')
print(f'improvement over Question 4: {W3 - (-res.fun):+.5f}')

In [ ]:
kT_3 = model.simulate(model.s_path(s0_3,phi_3,s_inf=sinf_3)).k[-1]
k_ss_sinf = model.steady_state(sinf_3)[0]
print(f'k_(T-1) under the best three-parameter rule = {kT_3:.4f}')
print(f'k* at s = s_inf = {sinf_3:.4f} is {k_ss_sinf:.4f}')

### Are they the same, and should they be?

They are essentially the same, and should be. With $T=100$ and $\varphi<1$, the savings rate has converged to $s_\infty$ well before the end, so the economy settles at the steady state belonging to $s_\infty$, and terminal capital equals the steady-state capital for that long-run savings rate. The optimizer picks an $s_\infty$ slightly below $\bar s=0.25$: a lower long-run savings rate gives a higher long-run consumption share, and with discounting the near-term gain outweighs ending with less capital.

## 2.6 Question 6

### An alternative shape for s_t

The geometric rule is one option. We use a rule that moves from $s_0$ to a long-run level $s_\infty$ along a logistic (S-shaped) transition instead of an exponential one:

$$ s_t = s_\infty + (s_0-s_\infty)\,\frac{1}{1+(t/t_0)^{p}} $$

It stays near $s_0$ for a while, then shifts to $s_\infty$ around $t_0$ with a sharpness set by $p$, and settles at the constant $s_\infty$, so the economy still ends in a steady state. We optimize over $(s_0,s_\infty,t_0,p)$.

In [ ]:
def s_path_logistic(model,s0,s_inf,t0,p):
    t = np.arange(model.par.T)
    s = s_inf + (s0-s_inf)/(1.0+(t/max(t0,1e-6))**p)
    return np.clip(s,0.0,1.0)

def evaluate_logistic(x):
    s0,s_inf,t0,p = x
    s = s_path_logistic(model,s0,s_inf,t0,p)
    sim = model.simulate(s)
    return -model.welfare(sim.c)

x0 = np.array([0.20, sinf_3, 5.0, 2.0])
res_log = optimize.minimize(evaluate_logistic, x0, method='Nelder-Mead',
                            options={'xatol':1e-6,'fatol':1e-10,'maxiter':5000})
s0_L,sinf_L,t0_L,p_L = res_log.x
W_log = -res_log.fun
print(f'logistic rule: s0 = {s0_L:.4f}, s_inf = {sinf_L:.4f}, t0 = {t0_L:.3f}, p = {p_L:.3f}')
print(f'W = {W_log:.4f}')

In [ ]:
summary = pd.DataFrame({
    'rule': ['Q4: two-parameter','Q5: three-parameter','Q6: logistic'],
    'W':    [-res.fun, W3, W_log],
})
summary['W - Q4'] = summary['W'] - (-res.fun)
display(summary.style.format({'W':'{:.4f}','W - Q4':'{:+.5f}'}).set_caption('Best welfare by rule'))

The three rules reach almost the same welfare. Once the rule can start low and settle at a freely chosen long-run level, the shape of the transition in between has little effect, because discounting puts most weight on the first few periods and all three rules consume heavily there. The logistic rule does not beat Question 5: the gains come from the endpoints, not the transition shape.

# 3. A portfolio with a risky and a safe asset

Wealth $W_t$ is split between a risky asset (gross return $R_t=\exp(\mu+\sigma\varepsilon_t)$)
and a safe asset (gross return $R^f=\exp(r)$). The investor has a target risky share
$\theta^*$ and a no-trade band of width $\Delta$: if $|\theta_t-\theta^*|>\Delta$ the portfolio
is traded back to $\theta^*$ at a proportional cost $\tau$, otherwise nothing happens. Only
terminal wealth matters, ranked by $E[u(W_T)]$ with CRRA utility.

`PortfolioModelClass` in `PortfolioModel.py` holds this. We filled in `trade`, `simulate` and
`summary`; the return draw and utility were given.

In [ ]:
from PortfolioModel import PortfolioModelClass
pm = PortfolioModelClass()
print(pm)

## 3.1 Question 1

### Draw the returns

We draw $\varepsilon_t$ for all periods and portfolios in one $(N,T)$ array with the given seed
and form $R_t$.

In [ ]:
R = pm.draw_returns()
print(f'R has shape {R.shape}')

logR = np.log(R)
print(f'mean log R = {logR.mean():.4f}  (mu = {pm.par.mu})')
print(f'std  log R = {logR.std():.4f}  (sigma = {pm.par.sigma})')
print(f'mean R     = {R.mean():.4f}  (exp(mu+0.5 sigma^2) = {np.exp(pm.par.mu+0.5*pm.par.sigma**2):.4f})')

The sample mean and standard deviation of $\log R_t$ match $\mu$ and $\sigma$, and the mean
of $R_t$ matches $\exp(\mu+\tfrac12\sigma^2)$, which is the mean of a lognormal. So the draws
are correct.

In [ ]:
Rf = np.exp(pm.par.r)

fig,axes = plt.subplots(1,2,figsize=(13,5))

# i. histogram of the gross return
axes[0].hist(R.ravel(), bins=80, color=colors[0], alpha=0.8)
axes[0].set_title('Gross return $R_t$'); axes[0].set_xlabel('$R_t$'); axes[0].set_ylabel('count')

# ii. value of 1 invested at t=0, 20 risky paths and the safe path, log y-axis
ax = axes[1]
paths = np.cumprod(R[:20,:], axis=1)
paths = np.column_stack([np.ones(20), paths])      # start every path at 1
for row in paths: ax.plot(row, color=colors[0], alpha=0.4, lw=0.9)
ax.plot(Rf**np.arange(pm.par.T+1), color='black', lw=2.5, label='safe asset')
ax.set_yscale('log')
ax.set_title('Value of 1 invested at t=0 (20 risky paths)')
ax.set_xlabel('t'); ax.set_ylabel('value (log scale)'); ax.legend()
fig.tight_layout()

## 3.2 Question 2

### The two extreme rules, no cost

With $\tau=0$ we simulate $\Delta=0$ (trade every period) and $\Delta=1$ (never trade) on the
same drawn returns, and report the six numbers.

In [ ]:
def six_numbers(**kwargs):
    m = PortfolioModelClass(**kwargs)
    m.simulate(R=R)                  # same drawn returns everywhere
    return m.summary()

labels = ['trades','distance','mean_WT','median_WT','p10_WT','EU']
names  = ['avg trades','avg distance','mean W_T','median W_T','10th pct W_T','E[u(W_T)]']

rules = {'Delta=0 (always)': six_numbers(Delta=0.0,tau=0.0),
         'Delta=1 (never)':  six_numbers(Delta=1.0,tau=0.0)}

tab = pd.DataFrame(rules, index=labels).rename(index=dict(zip(labels,names)))
display(tab.style.format('{:.4f}').set_caption('The two extreme rules, tau = 0'))

In [ ]:
m0 = PortfolioModelClass(Delta=0.0,tau=0.0); m0.simulate(R=R)
m1 = PortfolioModelClass(Delta=1.0,tau=0.0); m1.simulate(R=R)

fig,axes = plt.subplots(1,2,figsize=(13,5))

# i. terminal wealth histograms
ax = axes[0]
ax.hist(m0.sim.WT, bins=80, alpha=0.6, label='Delta=0', color=colors[0])
ax.hist(m1.sim.WT, bins=80, alpha=0.6, label='Delta=1', color=colors[1])
ax.set_xlim(0,30); ax.set_title('Terminal wealth $W_T$'); ax.set_xlabel('$W_T$'); ax.legend()

# ii. mean risky share over time with a 10-90 band
ax = axes[1]
for m,label,c in [(m0,'Delta=0',colors[0]),(m1,'Delta=1',colors[1])]:
    th = m.sim.theta[:,:-1]                        # drop the extra final column
    ax.plot(th.mean(axis=0), color=c, lw=2, label=label)
    ax.fill_between(np.arange(th.shape[1]),
                    np.percentile(th,10,axis=0), np.percentile(th,90,axis=0),
                    color=c, alpha=0.2)
ax.set_title('Risky share $\\theta_t$ (mean and 10-90 band)')
ax.set_xlabel('t'); ax.set_ylabel(r'$\theta_t$'); ax.legend()
fig.tight_layout()

### What happens to theta when the portfolio is never traded

With $\Delta=1$ nothing is traded, so the risky share drifts. The risky asset earns more on average, so it grows faster and $\theta_t$ moves up from the starting $0.5$, and the 10-90 band widens: some portfolios end almost fully in the risky asset, others fall back. With $\Delta=0$ the share is pulled back to $0.5$ every period, so the mean stays flat and the band stays tight.

## 3.3 Question 3

### A sweep over the band width

Now $\tau=0.01$. We report the six numbers for a range of $\Delta$, all on the same drawn
returns.

In [ ]:
Deltas = [0,0.025,0.05,0.075,0.10,0.15,0.20,0.30,1]
sweep = {D: six_numbers(Delta=D,tau=0.01) for D in Deltas}

tabD = pd.DataFrame(sweep).T
tabD.columns = names
tabD.index.name = 'Delta'
display(tabD.style.format('{:.4f}').set_caption('Six numbers by band width (tau = 0.01)'))

In [ ]:
EU = tabD['E[u(W_T)]']
tr = tabD['avg trades']
di = tabD['avg distance']

fig,axes = plt.subplots(1,2,figsize=(13,5))

ax = axes[0]
ax.plot(Deltas, tr, 'o-', color=colors[0], label='avg trades')
ax.set_xlabel('Delta'); ax.set_ylabel('avg trades', color=colors[0])
ax2 = ax.twinx(); ax2.grid(False)
ax2.plot(Deltas, di, 's-', color=colors[1], label='avg distance')
ax2.set_ylabel('avg distance', color=colors[1])
ax.set_title('Trading and drift vs Delta')

axes[1].plot(Deltas, EU, 'o-', color=colors[2])
axes[1].set_title('Expected utility vs Delta'); axes[1].set_xlabel('Delta'); axes[1].set_ylabel('E[u(W_T)]')
fig.tight_layout()

In [ ]:
best_D = EU.idxmax()
print(f'best Delta = {best_D}, E[u] = {EU[best_D]:.6f}')
print(f'beats Delta=0 by {EU[best_D]-EU[0]:+.6f}')
print(f'beats Delta=1 by {EU[best_D]-EU[1]:+.6f}')

### What happens as Delta grows

As $\Delta$ grows the portfolio trades less often and drifts further from the target, so the trade count falls steeply and the average distance rises. From $\Delta=0$ to the best $\Delta$, the trade count falls and the mean and spread of terminal wealth rise, because letting the risky share drift keeps more in the high-return asset and saves transaction costs; the median moves less. Expected utility is hump-shaped: a small band avoids paying to correct small deviations, while too wide a band lets risk build up, which a risk-averse investor dislikes.

Ranking by the mean of $W_T$ instead would give a different answer. The mean keeps rising as $\Delta\to 1$, because never trading maximizes exposure to the high-return asset, so ranking by the mean points to $\Delta=1$. Expected utility penalizes the extra risk and prefers an interior band.

# Notes

Problem 1 downloads from FRED when `fredapi.txt` is present, and otherwise uses the cached `data` folder. Problems 2 and 3 are self-contained given the two modules. The seed 2026 is used throughout Problem 3, and the same drawn returns are reused across rules.